[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jhavierc/NLP_taller_1/blob/main/notebook_taller_2_v2.ipynb)

> **Nota sobre Google Colab:** este cuaderno usa una sola GPU (a diferencia de taller 1/3, que usan 2). Corre igual en el T4 gratuito de Colab que en Kaggle.

# Taller 2 - NER clinico en espanol con un Transformer entrenado desde cero

## Integrantes

**Carlos Javier Cepeda, David Salamanca, Jose Milciades Ordoñez**

## Objetivo

Resolvemos el mismo problema de los talleres 1 y 3 (reconocimiento de entidades clinicas sobre el corpus SPACCC), pero esta vez implementamos y entrenamos **desde cero** un Transformer encoder (embeddings, positional encoding y multi-head attention escritos a mano, sin pesos preentrenados), siguiendo el patron de la guia de referencia `1-transformers-from-scratch.ipynb` de la Sesion 2 del curso -- adaptado de clasificacion de documentos a clasificacion por token (NER).

## Donde queda este taller en la progresion del repositorio

| Taller | Modelo | Tokenizacion | Pesos iniciales | Framework de entrenamiento |
|---|---|---|---|---|
| 1 | Bi-LSTM | spaCy (nivel palabra) | Aleatorios | Loop manual en PyTorch |
| **2 (este)** | **Transformer encoder (a mano)** | **BPE propio, entrenado sobre SPACCC** | **Aleatorios** | **PyTorch Lightning** |
| 3 | BERT / RoBERTa clinico | Subword preentrenado | **Preentrenados** | Hugging Face `Trainer` |

## Que tomamos de la guia de referencia (y que cambiamos)

- **Tokenizer BPE propio**: igual que la guia, re-entrenamos el algoritmo BPE de GPT-2 (`train_new_from_iterator`) sobre nuestro propio corpus -- aqui, los 600 documentos de entrenamiento de SPACCC, no noticias en espanol. Como es un tokenizer *fast*, podemos alinear las anotaciones BIO por offset de caracter igual que en talleres 1 y 3.
- **Positional encoding, atencion multi-cabeza y bloque Transformer**: reimplementamos las mismas piezas que la guia (`SinusoidPE`, `MultiHeadAttention`, `TransformerBlock`), mejorando los 4 bugs que la propia guia documenta: el assert de divisibilidad usa `%` en vez de `&`; las capas lineales usan el parametro `embed_size` y no una variable global; se corrige el typo `comibe_heads` -> `combine_heads`; y sobre todo, **agregamos las conexiones residuales** que la guia senala como faltantes (`LayerNorm(x + Sublayer(x))`), y permitimos apilar `num_layers` bloques.
- **Cabezal de clasificacion**: la guia usa `Flatten + Linear` para clasificar el documento completo (el 95% de sus parametros, segun su propio analisis). Como aqui clasificamos por **token** (NER), no hace falta ese `Flatten`: aplicamos un `Linear(embed_dim, num_etiquetas)` a cada posicion, igual de simple que el cabezal de la Bi-LSTM en taller 1.
- **Framework de entrenamiento**: usamos **PyTorch Lightning**, igual que la guia -- una tercera biblioteca de entrenamiento distinta al loop manual de taller 1 y al `Trainer` de Hugging Face de taller 3.
- **Las "dos variantes" de este taller**: en vez de con_pos/sin_pos (taller 1) o frozen/fine_tuned (taller 3), comparamos **`shallow`** (1 bloque Transformer) contra **`deep`** (4 bloques) -- es la mejora #4 que la propia guia sugiere ("apilar N bloques, con residuales arregladas, 2-4 capas"), y una pregunta empirica relevante dado que la guia concluye que los transformers desde cero son "hambrientos de datos".
- **Metrica**: `entity_spans`/`scores_from_counts`, la misma funcion de taller 1 (no `seqeval`: verificamos que su modo estricto descarta las transiciones BIO ilegales en vez de contarlas como prediccion incorrecta, lo que haria el F1 no comparable con taller 1).

## Aviso importante

Este cuaderno requiere GPU y **no se ha ejecutado todavia** (no hay GPU disponible en el entorno donde se escribio). Ejecuta las PASO 01 a 10 en Kaggle o Colab; la PASO 10 imprime la tabla comparativa que va en el informe final (al final del cuaderno, dejado como plantilla).

### PASO 01 - Dependencias

Instalamos `transformers` (solo para entrenar nuestro propio tokenizer BPE con `train_new_from_iterator`; no cargamos ningun modelo preentrenado) y `pytorch-lightning` (framework de entrenamiento). Pineamos las versiones exactas de Lightning que verificamos, porque es una libreria distinta a las de taller 1 y 3 y no queremos sorpresas de API entre versiones. Las metricas de NER las calculamos con la misma funcion `entity_spans` de taller 1 (PASO 07), no con `seqeval`, para que el F1 sea estrictamente comparable entre los tres talleres.

In [54]:
# PASO 01 - Dependencias (Internet activado; ejecutar una vez por sesion)
import os, sys, subprocess, importlib.util

def pip_install(*args):
    """Corre pip capturando su salida: si falla, imprime el error real de pip en
    vez de un CalledProcessError vacio (necesario porque -q oculta el traceback de
    fondo que explica CUAL paquete fallo y por que)."""
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-4000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f'pip install fallo (codigo {result.returncode}): {args}')

# Kaggle a veces trae un pip desactualizado que no sabe generar metadatos para
# paquetes "legacy" (setup.py sin pyproject.toml); actualizarlo primero evita el
# error "python setup.py egg_info did not run successfully".
pip_install('--upgrade', 'pip', 'setuptools', 'wheel')

packages = ['transformers>=5,<6', 'pytorch-lightning==2.6.6', 'torchmetrics==1.9.0',
            'pandas>=2,<3', 'pyarrow>=14', 'requests>=2.31', 'tqdm>=4.66']
if importlib.util.find_spec('torch') is None:
    packages.append('torch>=2.4,<3')
pip_install(*packages)
print('PASO 01 OK. Dependencias instaladas. Continua con la PASO 02.')

PASO 01 OK. Dependencias instaladas. Continua con la PASO 02.


### PASO 02 - Configuracion, GPU y reproducibilidad

Definimos los hiperparametros del Transformer (dimension de embedding, cabezas de atencion, tamano de la red feed-forward, dropout), las dos variantes de profundidad (`shallow`=1 bloque, `deep`=4 bloques), las tres semillas y las mismas etiquetas BIO de talleres 1 y 3. A diferencia de esos dos, aqui solo pedimos **una GPU**: un encoder de este tamano, entrenado sobre 600 documentos, no la necesita, y el multi-GPU de Lightning via DDP es fragil dentro de un notebook.

In [ ]:
# PASO 02 - Hiperparametros del Transformer, dos profundidades, tres semillas
import os, json, time, random, math, hashlib, platform, traceback
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import requests, torch
from torch import nn

CFG = dict(
    split_seed=42, training_seeds=[42, 123, 2026], fraction=1.0, validation_fraction=0.20,
    vocab_size=8000, max_length=192, embed_dim=128, num_heads=8, ff_dim=512, dropout=0.2,
    variants={'shallow': 1, 'deep': 4},  # num_layers por variante
    batch_size=16, learning_rate=3e-4, weight_decay=0.01, epochs=30, patience=5,
    class_weights=True,
    run_final_test=True, final_test_variant=None, final_test_seed=None,
    requested_gpus=1, allow_cpu_for_debug=False,
)
assert 0 < CFG['fraction'] <= 1
assert 0 < CFG['validation_fraction'] < 1, 'Usa validation_fraction=0.20 para reservar el 20 %.'
assert isinstance(CFG['split_seed'], int) and 0 <= CFG['split_seed'] < 2**32
assert (isinstance(CFG['training_seeds'], list) and len(CFG['training_seeds']) >= 2
        and all(type(s) is int and 0 <= s < 2**32 for s in CFG['training_seeds'])
        and len(set(CFG['training_seeds'])) == len(CFG['training_seeds'])), 'Usa al menos dos semillas enteras distintas.'
assert CFG['embed_dim'] % CFG['num_heads'] == 0, 'embed_dim debe ser divisible por num_heads.'
random.seed(CFG['split_seed']); np.random.seed(CFG['split_seed']); torch.manual_seed(CFG['split_seed'])
assert CFG['requested_gpus'] == 1, 'Este cuaderno esta pensado para 1 GPU; usa taller 1/3 como referencia de multi-GPU.'
available_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if available_gpus == 0 and CFG['allow_cpu_for_debug']:
    device = torch.device('cpu')
    print('AVISO: modo de depuracion CPU explicito; el entrenamiento real necesita GPU.')
elif available_gpus < CFG['requested_gpus']:
    raise RuntimeError(f'PASO 02: se requiere {CFG["requested_gpus"]} GPU CUDA y se detectaron '
                       f'{available_gpus}. Activa GPU en Kaggle/Colab y reinicia la sesion.')
else:
    device = torch.device('cuda:0')
    torch.cuda.set_device(device)
    torch.cuda.manual_seed_all(CFG['split_seed'])

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORK = ROOT / 'spaccc_transformer_scratch'
WORK.mkdir(parents=True, exist_ok=True)
RUN = WORK / (time.strftime('%Y%m%d_%H%M%S') + '_' + str(time.time_ns())[-6:])
RUN.mkdir()
CACHE = WORK / 'cache'; CACHE.mkdir(exist_ok=True)
LABELS = ['CHEMICAL', 'DISEASE', 'PROCEDURE', 'PROTEIN', 'SYMPTOM']
TAGS = ['O'] + [f'{prefix}-{label}' for label in LABELS for prefix in ('B', 'I')]
tag_to_ix = {tag: i for i, tag in enumerate(TAGS)}
IGNORE = -100  # Misma convencion que talleres 1 y 3: padding y subwords no evaluables.
START = time.perf_counter()
REPORT = {'config': CFG.copy(), 'run_dir': str(RUN), 'device': str(device),
          'gpu': {'available_count': available_gpus, 'requested': CFG['requested_gpus'],
                  'devices': [{'id': i, 'name': torch.cuda.get_device_name(i),
                               'total_memory_gb': torch.cuda.get_device_properties(i).total_memory / 2**30}
                              for i in range(available_gpus)]},
          'cpu': {'logical_cores': os.cpu_count(), 'machine': platform.machine()},
          'versions': {'python': platform.python_version(), 'torch': str(torch.__version__)}}
(RUN / 'config.json').write_text(json.dumps(REPORT, indent=2), encoding='utf-8')
print('PASO 02 OK:', json.dumps(REPORT, indent=2))

PASO 02 OK: {
  "config": {
    "split_seed": 42,
    "training_seeds": [
      42,
      123,
      2026
    ],
    "fraction": 1.0,
    "validation_fraction": 0.2,
    "vocab_size": 8000,
    "max_length": 192,
    "embed_dim": 128,
    "num_heads": 8,
    "ff_dim": 512,
    "dropout": 0.2,
    "variants": {
      "shallow": 1,
      "deep": 4
    },
    "batch_size": 16,
    "learning_rate": 0.0003,
    "weight_decay": 0.01,
    "epochs": 30,
    "patience": 5,
    "class_weights": true,
    "run_final_test": false,
    "final_test_variant": null,
    "final_test_seed": null,
    "requested_gpus": 1,
    "allow_cpu_for_debug": false
  },
  "run_dir": "/kaggle/working/spaccc_transformer_scratch/20260919_165308_906088",
  "device": "cuda:0",
  "gpu": {
    "available_count": 2,
    "requested": 1,
    "devices": [
      {
        "id": 0,
        "name": "Tesla T4",
        "total_memory_gb": 14.56219482421875
      },
      {
        "id": 1,
        "name": "Tesla T4",
        "tota

### PASO 03 - Descarga y auditoria de los datos

Reutilizamos exactamente el mismo procedimiento de talleres 1 y 3: descargamos y dejamos en cache las anotaciones NER, los documentos completos y el conjunto de prueba desde Hugging Face (mismo corpus SPACCC). Esta PASO audita que la descarga sea consistente; el analisis exploratorio completo (mismo corpus que taller 1) queda en la PASO 04.

In [56]:
# PASO 03 - Descarga, textos completos y auditoria de todas las anotaciones (igual que talleres 1 y 3)
def download_parquet(repo, split, name):
    """Descarga el parquet `split` del dataset `repo` (Hugging Face) y lo guarda como
    `name` dentro de la carpeta de cache. Si el archivo ya existe en cache lo reutiliza
    sin volver a descargarlo. Reintenta la descarga hasta 3 veces y valida que el
    parquet descargado se pueda leer antes de reemplazar la cache."""
    path = CACHE / name
    if not path.exists():
        url = f'https://huggingface.co/datasets/{repo}/resolve/main/data/{split}-00000-of-00001.parquet'
        print('Descargando:', name, flush=True)
        error = None
        for attempt in range(3):
            try:
                response = requests.get(url, timeout=(15, 120))
                response.raise_for_status()
                tmp = path.with_suffix('.tmp')
                tmp.write_bytes(response.content)
                pd.read_parquet(tmp)  # No guardar una respuesta invalida como cache.
                tmp.replace(path)
                break
            except Exception as exc:
                error = exc
                print(f'Intento {attempt + 1}/3: {type(exc).__name__}: {exc}', flush=True)
        if not path.exists():
            raise RuntimeError('PASO 03: revisa Internet de Kaggle y comparte este error.') from error
    return pd.read_parquet(path)

t0 = time.perf_counter()
df_train = download_parquet('IEETA/SPACCC-Spanish-NER', 'train', 'annotations_train.parquet')
df_test = download_parquet('IEETA/SPACCC-Spanish-NER', 'test', 'annotations_test.parquet')
df_docs = download_parquet('IEETA/SPACCC-documents', 'train', 'documents.parquet')
def document_id(value):
    """Obtiene el identificador de un documento a partir de su nombre de archivo."""
    return Path(str(value)).stem.removeprefix('es-')
required = {'filename', 'label', 'start_span', 'end_span', 'text'}
for frame in (df_train, df_test):
    assert required <= set(frame.columns), 'Columnas de anotacion inesperadas.'
    assert not frame[list(required)].isnull().any().any(), 'Anotaciones con valores nulos.'
    frame['doc_id'] = frame.filename.map(document_id)
    assert set(frame.label) <= set(LABELS), 'Categorias nuevas: revisar TAGS.'
df_docs['doc_id'] = df_docs.filename.map(document_id)
assert not df_docs.doc_id.duplicated().any(), 'Identificadores de documento duplicados.'
assert df_docs.document.map(lambda x: isinstance(x, str) and bool(x)).all()
texts = dict(zip(df_docs.doc_id, df_docs.document))
official_train_ids = sorted(df_train.doc_id.unique())
official_test_ids = sorted(df_test.doc_id.unique())
assert not set(official_train_ids) & set(official_test_ids), 'Fuga: documentos en train y test.'
assert (set(official_train_ids) | set(official_test_ids)) <= texts.keys(), 'Faltan documentos completos.'
bad_offsets, quote_only_differences = [], []
for split, frame in [('train', df_train), ('test', df_test)]:
    for row in frame.itertuples(index=False):
        a, b = int(row.start_span), int(row.end_span)
        actual = texts[row.doc_id][a:b]
        detail = {'split': split, 'doc_id': row.doc_id, 'start': a, 'end': b}
        if not (0 <= a < b <= len(texts[row.doc_id])):
            bad_offsets.append(detail)
        elif actual != row.text:
            if actual.replace(chr(34), '') == row.text.replace(chr(34), ''):
                quote_only_differences.append(detail)
            else:
                bad_offsets.append(detail)
assert not bad_offsets, f'Offsets incompatibles: {len(bad_offsets)}. Ejemplos: {bad_offsets[:5]}'
fingerprint = lambda s: hashlib.sha256(' '.join(s.split()).encode()).hexdigest()
train_hashes = {fingerprint(texts[k]) for k in official_train_ids}
test_hashes = {fingerprint(texts[k]) for k in official_test_ids}
assert not train_hashes & test_hashes, 'Textos duplicados entre train y test: revisar particion.'
REPORT['data'] = {'train_documents': len(official_train_ids), 'test_documents': len(official_test_ids),
                  'train_annotations': len(df_train), 'test_annotations': len(df_test),
                  'offset_errors': len(bad_offsets), 'quote_only_differences': quote_only_differences,
                  'download_audit_seconds': time.perf_counter() - t0,
                  'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                             for p in CACHE.glob('*.parquet')}}
print('PASO 03 OK:', json.dumps(REPORT['data'], indent=2))

PASO 03 OK: {
  "train_documents": 750,
  "test_documents": 250,
  "train_annotations": 33757,
  "test_annotations": 11239,
  "offset_errors": 0,
  "quote_only_differences": [
    {
      "split": "test",
      "doc_id": "S0004-06142008000700003-1",
      "start": 1560,
      "end": 1647
    },
    {
      "split": "test",
      "doc_id": "S1134-80462005000100007-1",
      "start": 127,
      "end": 145
    }
  ],
  "download_audit_seconds": 0.3456549940001423,
  "sha256": {
    "annotations_train.parquet": "76c692d7cc49d35d030aeb5c27523862ebd7585bdcf31e596f2f7d6ebe37c033",
    "documents.parquet": "d866b787254d0ebdc4e7453ea3fdce25edfb1fc058b536404d1dcf15e5a40f24",
    "annotations_test.parquet": "53bf36209410f657f773cfeb804b08487ab1e8d23e3af9e972cc3927140e1413"
  }
}


### PASO 04 - Analisis exploratorio de los datos (EDA)

Antes de tokenizar, exploramos el corpus para entender su composicion: distribucion de categorias, longitud de las entidades y densidad de anotaciones por documento. Es el mismo analisis de taller 1 (mismo corpus SPACCC), incluido aqui para que este cuaderno sea autocontenido.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 1. Descubrir los nombres reales de las columnas
print("Columnas disponibles en el DataFrame:", df_train.columns.tolist())
display(df_train.head(2))

In [ ]:
# 1. Inspeccion general
print("--- PRIMERAS 5 FILAS ---")
display(df_train.head())

print("\n--- VALORES NULOS ---")
print(df_train.isnull().sum())

# 2. Distribucion de las clases / etiquetas de entidades
print("\n--- FRECUENCIA DE ETIQUETAS (LABELS) ---")
print(df_train['label'].value_counts())

plt.figure(figsize=(10, 5))
sns.countplot(
    data=df_train,
    x='label',
    hue='label',
    order=df_train['label'].value_counts().index,
    palette='viridis',
    legend=False
)
plt.title('Distribucion de Categorias de Entidades Clinicas')
plt.xlabel('Etiqueta')
plt.ylabel('Cantidad de Apariciones')
plt.xticks(rotation=45)
plt.show()

# 3. Analisis del tamano de las entidades (numero de palabras por entidad)
df_train['entity_word_count'] = df_train['text'].apply(lambda x: len(str(x).split()))

print("\n--- ESTADISTICAS DE PALABRAS POR ENTIDAD ---")
print(df_train['entity_word_count'].describe())

plt.figure(figsize=(10, 5))
sns.histplot(df_train['entity_word_count'], bins=15, kde=True, color='teal')
plt.title('Distribucion de la Longitud de las Entidades (Numero de palabras)')
plt.xlabel('Cantidad de Palabras en la Entidad')
plt.ylabel('Frecuencia')
plt.show()

# 4. Analisis de densidad: cuantas entidades aparecen por cada documento clinico?
entities_per_file = df_train.groupby('filename').size()

print("\n--- ESTADISTICAS DE ENTIDADES POR DOCUMENTO ---")
print(entities_per_file.describe())

plt.figure(figsize=(10, 5))
sns.histplot(entities_per_file, bins=30, kde=True, color='purple')
plt.title('Distribucion de Entidades por Reporte Clinico')
plt.xlabel('Cantidad de Entidades en el Documento')
plt.ylabel('Frecuencia de Documentos')
plt.show()

# Verificacion estructural superficial
print("--- VERIFICACION ESTRUCTURAL: TEST VS TRAIN ---")

# 1. Comparar columnas
columns_match = list(df_train.columns) == list(df_test.columns)
print(f"Las columnas son identicas?: {columns_match}")
print(f"Columnas en Train: {df_train.columns.tolist()}")
print(f"Columnas en Test:  {df_test.columns.tolist()}")

# 2. Comparar tipos de datos (dtypes)
print("\n--- TIPOS DE DATOS EN TRAIN ---")
print(df_train.dtypes)

print("\n--- TIPOS DE DATOS EN TEST ---")
print(df_test.dtypes)

# 3. Verificacion de dimensiones
print(f"\nDimensiones Train (filas, columnas): {df_train.shape}")
print(f"Dimensiones Test (filas, columnas):  {df_test.shape}")

- Distribucion de Clases: encontramos un total de 33,757 entidades clinicas distribuidas en cinco categorias principales. PROCEDURE encabeza con 11,065 apariciones, seguida por SYMPTOM (9,091) y DISEASE (8,065). Las clases minoritarias corresponden a CHEMICAL (3,283) y PROTEIN (2,253). No encontramos valores nulos en ninguna columna, lo que valida la integridad de los datos descargados.

- La distribucion de entidades presenta desbalance: PROCEDURE concentra el 32.78 % de las anotaciones, mientras PROTEIN representa solo el 6.68 %. Este desbalance puede favorecer el aprendizaje de las clases mayoritarias y reducir el recall de las minoritarias. Por eso usamos pesos de clase en la funcion de perdida (PASO 07) y reportamos metricas independientes por categoria.

- Longitud de las Entidades: observamos que la gran mayoria de las entidades medicas se concentran entre 1 y 3 palabras (con una media de 3.03 tokens por entidad). Sin embargo, existen casos complejos que alcanzan hasta 44 palabras. Esta variabilidad es relevante para el Transformer que construimos en este taller: la atencion multi-cabeza puede relacionar directamente el inicio y el final de una entidad larga sin importar la distancia entre esas posiciones (a diferencia de una red recurrente), aunque el tamano de bloque (`max_length`) sigue acotando cuantas palabras puede abarcar una sola entidad.

- Densidad de Informacion por Documento: observamos un promedio de 45 entidades clinicas por cada reporte, con documentos que alcanzan un maximo de 183 menciones en un total de 750 archivos analizados. Esto confirma una alta densidad informativa por historia clinica, por lo que necesitamos que nuestro `DataLoader` gestione bloques de texto amplios durante la tokenizacion con nuestro tokenizer BPE propio (PASO 06).

- Con la verificacion estructural comprobamos que ambos conjuntos de datos, entrenamiento y prueba, son totalmente compatibles:

  Las columnas de entrenamiento y prueba son identicas (filename, ann_id, label, start_span, end_span, text).

  Los tipos de datos coinciden de manera exacta en todos los campos, lo que nos garantiza que el pipeline de transformacion que aplicamos al conjunto de entrenamiento funciona de forma identica en el conjunto de prueba sin errores de tipo.

  La proporcion de los datos es coherente para un split de validacion/prueba estandar en machine learning: contamos con 33,757 registros de entrenamiento y 11,239 registros de prueba.

  Con esta validacion completada y libre de fugas de datos (data leakage), dejamos preparado el terreno para estructurar la transformacion de los textos clinicos a nivel de documento y token.

### PASO 05 - Particion fija por documento

Usamos exactamente la misma particion de talleres 1 y 3: `split_seed=42`, division por documento completo, 600 documentos de train y 150 de validacion. Verificamos la misma huella SHA-256 para confirmar que los 6 entrenamientos (2 variantes x 3 semillas) usan exactamente los mismos documentos que los otros dos talleres.

In [57]:
# PASO 05 - Particion unica compartida por las dos variantes (sin test)
rng = np.random.default_rng(CFG['split_seed'])
order = rng.permutation(official_train_ids).tolist()
n_sample = min(len(order), max(5, math.ceil(len(order) * CFG['fraction'])))
selected = order[:n_sample]
n_val = max(1, round(n_sample * CFG['validation_fraction']))
val_ids, train_ids = sorted(selected[:n_val]), sorted(selected[n_val:])
assert train_ids and val_ids
assert not set(train_ids) & set(val_ids)
assert not {fingerprint(texts[k]) for k in train_ids} & {fingerprint(texts[k]) for k in val_ids}
split_payload = {'train': train_ids, 'validation': val_ids, 'official_test': official_test_ids}
split_sha256 = hashlib.sha256(json.dumps(split_payload, sort_keys=True).encode()).hexdigest()
REPORT['split'] = {'sample_documents': n_sample, 'train_documents': len(train_ids),
                   'validation_documents': len(val_ids), 'test_used_for_training': False,
                   'split_seed': CFG['split_seed'], 'sha256': split_sha256}
(RUN / 'split.json').write_text(json.dumps({**split_payload, 'split_seed': CFG['split_seed'], 'sha256': split_sha256}, indent=2), encoding='utf-8')
print('PASO 05 - MUESTRA:', json.dumps(REPORT['split'], indent=2))
print('Anotaciones por categoria en la muestra:')
print(df_train[df_train.doc_id.isin(selected)].label.value_counts().to_string())
print('PASO 05 OK. Las 2 variantes x 3 semillas usaran estos mismos documentos; todavia no comenzo a entrenar.')
TALLER1_SPLIT_SHA256 = 'ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497'
if split_sha256 == TALLER1_SPLIT_SHA256:
    print('Huella de particion identica a talleres 1 y 3: los resultados son comparables entre los tres.')
else:
    print('AVISO: la huella de particion no coincide con la de taller 1 '
          f'(actual={split_sha256}, taller1={TALLER1_SPLIT_SHA256}).')

PASO 04 - MUESTRA: {
  "sample_documents": 750,
  "train_documents": 600,
  "validation_documents": 150,
  "test_used_for_training": false,
  "split_seed": 42,
  "sha256": "ab06b4e8dd54e120c2fed93f276f1944a6e67c5227281778c38add2eea4dc497"
}
Anotaciones por categoria en la muestra:
label
PROCEDURE    11065
SYMPTOM       9091
DISEASE       8065
CHEMICAL      3283
PROTEIN       2253
PASO 04 OK. Las 2 variantes x 3 semillas usaran estos mismos documentos; todavia no comenzo a entrenar.
Huella de particion identica a talleres 1 y 3: los resultados son comparables entre los tres.


### PASO 06 - Tokenizer BPE propio y alineacion BIO

Igual que el Paso 4 de la guia de referencia, re-entrenamos el algoritmo BPE byte-level de GPT-2 (`train_new_from_iterator`) para obtener un vocabulario propio -- pero aqui sobre los documentos de **entrenamiento** del corpus SPACCC (nunca sobre validacion, mismo principio de taller 1: "vocabulario solo de train"). Como es un tokenizer *fast*, alineamos las anotaciones a nivel de subword con la misma politica estricta y de resolucion de solapes que taller 1 y taller 3. A diferencia de taller 3 (BERT), aqui no agregamos `[CLS]`/`[SEP]`: esta arquitectura no los necesita, solo un token de padding para armar lotes.

In [58]:
# PASO 06 - Entrenamos un tokenizer BPE propio y alineamos las etiquetas BIO
from transformers import AutoTokenizer
from tqdm.auto import tqdm

t0 = time.perf_counter()
base_tokenizer = AutoTokenizer.from_pretrained('gpt2')

def corpus_iterator(doc_ids, batch_size=8):
    """Generador de lotes de texto para entrenar el tokenizer BPE (igual que el Paso 4
    de la guia de referencia), pero sobre los documentos de ENTRENAMIENTO de SPACCC en
    vez de noticias en espanol -- asi el vocabulario nunca ve texto de validacion."""
    for start in range(0, len(doc_ids), batch_size):
        yield [texts[doc_id] for doc_id in doc_ids[start:start + batch_size]]

tokenizer = base_tokenizer.train_new_from_iterator(
    corpus_iterator(train_ids), vocab_size=CFG['vocab_size'],
    length=math.ceil(len(train_ids) / 8), new_special_tokens=['[PAD]'])
tokenizer.pad_token = '[PAD]'
assert tokenizer.is_fast, 'Se necesita un tokenizer fast (con offset_mapping) para alinear BIO por subword.'
PAD_ID = tokenizer.pad_token_id
REPORT['tokenizer'] = {'vocab_size': len(tokenizer), 'pad_token_id': PAD_ID, 'base': 'gpt2 (re-entrenado desde cero)'}
print('PASO 06a OK. Tokenizer propio entrenado:', json.dumps(REPORT['tokenizer'], indent=2))

def resolve_gold(text, offset_mapping, subset):
    """Asigna una etiqueta BIO a cada subword de un documento a partir de sus offsets
    de caracter (inicio, fin) y las anotaciones de ese documento. Misma politica que
    talleres 1 y 3: alineacion ESTRICTA (el limite de la anotacion debe coincidir
    exactamente con el limite de algun subword) y, si dos anotaciones alineadas se
    solapan, se conserva la mas larga (y, en empate, la que empieza antes). Los
    subwords de una anotacion descartada quedan en IGNORE en vez de 'O', salvo que
    otra entidad retenida ya les haya asignado una etiqueta valida.

    NOTA: nuestro tokenizer BPE byte-level (estilo GPT-2) incluye el espacio previo
    dentro del offset del propio token (p. ej. el token de "fiebre" en "...con fiebre"
    reporta un offset que empieza en el espacio, no en la 'f'). Las anotaciones de
    SPACCC, en cambio, empiezan en la primera letra. Sin corregir esto, casi ninguna
    entidad alinea (se verifico empiricamente: 96% de las anotaciones quedaban como
    'unaligned'). Por eso recortamos los espacios iniciales del rango de cada subword
    antes de indexarlo en `starts`."""
    n = len(offset_mapping)
    starts, ends = {}, {}
    for idx, (s, e) in enumerate(offset_mapping):
        if s == e:
            continue
        content_start = s
        while content_start < e and text[content_start].isspace():
            content_start += 1
        starts.setdefault(content_start, idx)
        ends[e] = idx
    gold = [0] * n
    candidates, ignored_spans, seen = [], [], set()
    stats = Counter()
    for row in subset.itertuples(index=False):
        a, b, label = int(row.start_span), int(row.end_span), row.label
        stats['annotations'] += 1
        identity = (a, b, label)
        if identity in seen:
            stats['duplicates'] += 1
            continue
        seen.add(identity)
        if a not in starts or b not in ends or ends[b] < starts[a]:
            stats['unaligned'] += 1
            ignored_spans.append((a, b))
            continue
        candidates.append((a, b, label, starts[a], ends[b] + 1))
    for a, b, label, i, j in sorted(candidates, key=lambda x: (-(x[1] - x[0]), x[0], x[2])):
        if any(gold[k] != 0 for k in range(i, j)):
            stats['overlap_excluded'] += 1
            ignored_spans.append((a, b))
            continue
        gold[i] = tag_to_ix[f'B-{label}']
        gold[i + 1:j] = [tag_to_ix[f'I-{label}']] * (j - i - 1)
        stats['retained'] += 1
    for k, (s, e) in enumerate(offset_mapping):
        if gold[k] == 0 and any(s < b2 and e > a2 for a2, b2 in ignored_spans):
            gold[k] = IGNORE
    return gold, stats

def make_examples(ids, annotations, split_name):
    """Tokeniza los documentos `ids` con nuestro tokenizer BPE propio, alinea las
    anotaciones a nivel de subword y divide cada documento en bloques de a lo sumo
    `CFG['max_length']` subwords sin cortar ninguna entidad a la mitad -- misma logica
    de bloques que talleres 1 y 3. A diferencia de taller 3 no reservamos posiciones
    para tokens especiales: esta arquitectura solo necesita padding."""
    subset_all = annotations[annotations.doc_id.isin(ids)]
    # 'policy' incluye una version del algoritmo de alineacion: al subirla invalidamos
    # la cache vieja automaticamente (necesario porque corregimos un bug de alineacion
    # que no cambiaba ids/vocab_size/textos/anotaciones, solo la logica de resolve_gold).
    key = hashlib.sha256(json.dumps({'ids': ids, 'max_length': CFG['max_length'],
        'vocab_size': CFG['vocab_size'], 'policy': 'flat-longest-strict-strip-leading-space-v2',
        'texts': [fingerprint(texts[k]) for k in ids],
        'annotations': subset_all[['doc_id', 'start_span', 'end_span', 'label']].to_json()},
        sort_keys=True).encode()).hexdigest()
    cache_file = CACHE / f'features_{key}.json'
    start = time.perf_counter()
    if cache_file.exists():
        payload = json.loads(cache_file.read_text(encoding='utf-8'))
        payload['stats']['cache_hit'] = True
        payload['stats']['seconds_this_run'] = time.perf_counter() - start
        (RUN / f'alignment_{split_name}.json').write_text(json.dumps(payload['stats'], indent=2), encoding='utf-8')
        return payload['examples'], payload['stats']
    groups = {k: g for k, g in subset_all.groupby('doc_id')}
    empty_subset = subset_all.iloc[0:0]
    examples = []
    stats = Counter(documents=len(ids), annotations=0, duplicates=0, unaligned=0,
                    overlap_excluded=0, retained=0, tokens=0, ignored_tokens=0)
    for doc_id in tqdm(ids, desc=f'Tokenizando {split_name}'):
        encoding = tokenizer(texts[doc_id], add_special_tokens=False, return_offsets_mapping=True)
        input_ids, offset_mapping = encoding['input_ids'], encoding['offset_mapping']
        assert len(input_ids) > 0
        gold, doc_stats = resolve_gold(texts[doc_id], offset_mapping, groups.get(doc_id, empty_subset))
        for k, v in doc_stats.items():
            stats[k] += v
        n = len(input_ids)
        stats['tokens'] += n
        stats['ignored_tokens'] += gold.count(IGNORE)
        start_token = 0
        while start_token < n:
            end = min(start_token + CFG['max_length'], n)
            # Si el siguiente subword es I-, el corte cae dentro de una entidad: retrocedemos
            # hasta su inicio (o, si eso vacia el bloque, avanzamos hasta que termine).
            if end < n and gold[end] >= 0 and TAGS[gold[end]].startswith('I-'):
                cut = end
                while cut > start_token and gold[cut] >= 0 and TAGS[gold[cut]].startswith('I-'):
                    cut -= 1
                if cut > start_token:
                    end = cut
                else:
                    while end < n and gold[end] >= 0 and TAGS[gold[end]].startswith('I-'):
                        end += 1
            assert end > start_token
            examples.append({'doc_id': doc_id, 'input_ids': input_ids[start_token:end],
                             'attention_mask': [1] * (end - start_token), 'labels': gold[start_token:end]})
            start_token = end
    stats = dict(stats)
    stats.update(cache_hit=False, sequences=len(examples), seconds_this_run=time.perf_counter() - start)
    assert stats.get('retained', 0) > 0, 'No quedaron entidades. Comparte este error.'
    payload = {'examples': examples, 'stats': stats}
    cache_file.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
    (RUN / f'alignment_{split_name}.json').write_text(json.dumps(stats, indent=2), encoding='utf-8')
    return examples, stats

train_examples, train_stats = make_examples(train_ids, df_train, 'train')
val_examples, val_stats = make_examples(val_ids, df_train, 'validation')
REPORT['alignment'] = {'train': train_stats, 'validation': val_stats}
REPORT['preprocessing_seconds'] = time.perf_counter() - t0
print('train', json.dumps(train_stats, indent=2))
print('validation', json.dumps(val_stats, indent=2))
print('PASO 06 OK. Bloques:', len(train_examples), 'train,', len(val_examples), 'validacion.')




PASO 05a OK. Tokenizer propio entrenado: {
  "vocab_size": 8000,
  "pad_token_id": 1,
  "base": "gpt2 (re-entrenado desde cero)"
}


Tokenizando train:   0%|          | 0/600 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1270 > 1024). Running this sequence through the model will result in indexing errors


Tokenizando validation:   0%|          | 0/150 [00:00<?, ?it/s]

train {
  "documents": 600,
  "annotations": 26849,
  "duplicates": 0,
  "unaligned": 77,
  "overlap_excluded": 4286,
  "retained": 22486,
  "tokens": 300860,
  "ignored_tokens": 1485,
  "cache_hit": false,
  "sequences": 1883,
  "seconds_this_run": 1.534887257000264
}
validation {
  "documents": 150,
  "annotations": 6908,
  "duplicates": 0,
  "unaligned": 25,
  "overlap_excluded": 986,
  "retained": 5897,
  "tokens": 80944,
  "ignored_tokens": 473,
  "cache_hit": false,
  "sequences": 501,
  "seconds_this_run": 0.3806333749998885
}
PASO 05 OK. Bloques: 1883 train, 501 validacion.


### PASO 07 - Dataset de PyTorch, padding dinamico y pesos por clase

Igual que taller 1 (y a diferencia de taller 3, que usa `datasets`+`DataCollatorForTokenClassification` de Hugging Face), aqui escribimos un `Dataset` de PyTorch y una funcion `collate` a mano, con padding dinamico por lote. Mantenemos los pesos por clase de los otros dos talleres para compensar el desbalance hacia la etiqueta `O`.

In [59]:
# PASO 07 - Dataset de PyTorch, padding dinamico y pesos por clase
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class ClinicalDataset(Dataset):
    """Envuelve los bloques ya tokenizados y los convierte a tensores de indices de
    vocabulario y etiquetas BIO (igual espiritu que `ClinicalDataset` en taller 1)."""
    def __init__(self, examples): self.examples = examples
    def __len__(self): return len(self.examples)
    def __getitem__(self, idx):
        e = self.examples[idx]
        return (torch.tensor(e['input_ids'], dtype=torch.long),
                torch.tensor(e['attention_mask'], dtype=torch.long),
                torch.tensor(e['labels'], dtype=torch.long))

def collate(batch):
    """Junta una lista de bloques de distinta longitud en un solo lote con padding
    dinamico (cada lote se rellena solo hasta su secuencia mas larga)."""
    ids, masks, labels = zip(*batch)
    return {'input_ids': pad_sequence(ids, batch_first=True, padding_value=PAD_ID),
            'attention_mask': pad_sequence(masks, batch_first=True, padding_value=0),
            'labels': pad_sequence(labels, batch_first=True, padding_value=IGNORE)}

train_dataset, val_dataset = ClinicalDataset(train_examples), ClinicalDataset(val_examples)
label_counts = Counter(y for e in train_examples for y in e['labels'] if y != IGNORE)
assert label_counts[0] > 0 and sum(v for k, v in label_counts.items() if k > 0) > 0
class_weights_tensor = torch.ones(len(TAGS))
if CFG['class_weights']:
    freq = torch.tensor([label_counts[i] for i in range(len(TAGS))], dtype=torch.float)
    present = freq > 0
    class_weights_tensor[present] = (freq[present].sum() / (present.sum() * freq[present])).sqrt().clamp(0.25, 5.0)
REPORT['vocabulary'] = {'train_tag_counts': {TAGS[k]: v for k, v in sorted(label_counts.items())},
    'missing_train_tags': [TAGS[i] for i in range(len(TAGS)) if label_counts[i] == 0],
    'train_blocks': len(train_examples), 'validation_blocks': len(val_examples)}
print('PASO 07 OK:', json.dumps(REPORT['vocabulary'], indent=2))
if REPORT['vocabulary']['missing_train_tags']:
    print('AVISO: faltan etiquetas en esta muestra; el test no permitira valorar esas etiquetas.')

PASO 06 OK: {
  "train_tag_counts": {
    "O": 194651,
    "B-CHEMICAL": 1917,
    "I-CHEMICAL": 3377,
    "B-DISEASE": 5873,
    "I-DISEASE": 21955,
    "B-PROCEDURE": 7255,
    "I-PROCEDURE": 25974,
    "B-PROTEIN": 1118,
    "I-PROTEIN": 1660,
    "B-SYMPTOM": 6323,
    "I-SYMPTOM": 29272
  },
  "missing_train_tags": [],
  "train_blocks": 1883,
  "validation_blocks": 501
}


### PASO 08 - Transformer encoder desde cero (Lightning) y metricas de NER

Implementamos las piezas de la guia de referencia -- `SinusoidPE`, `MultiHeadAttention`, `TransformerBlock` -- corrigiendo sus 4 bugs documentados (assert con `%`, uso del parametro en vez de una variable global, typo `combine_heads`, y **conexiones residuales agregadas**), y las apilamos en un `TransformerEncoder` de `num_layers` bloques (1 para `shallow`, 4 para `deep`). El cabezal de clasificacion es un simple `Linear` por token (no hace falta `Flatten`, porque no clasificamos el documento completo sino cada palabra). Envolvemos todo en un `LightningModule` que entrena con perdida ponderada por clase y evalua con `entity_spans`/`scores_from_counts` (coincidencia EXACTA de limites y categoria) -- la misma funcion de taller 1, para que el F1 sea estrictamente comparable entre los tres talleres.

In [60]:
# PASO 08 - Transformer encoder desde cero, envuelto en un LightningModule
import pytorch_lightning as pl

def entity_spans(tag_ids):
    """Agrupa una secuencia de indices de etiqueta BIO en entidades completas
    (inicio, fin, categoria) -- identica a la de taller 1 (y a la del taller 2 de
    Jose): una I- sin entidad activa de la misma categoria se trata como inicio de
    una entidad nueva (reparacion deterministica), contada aparte en `invalid`. La
    usamos en vez de `seqeval` para que el F1 sea estrictamente comparable entre los
    tres talleres: `seqeval` en modo estricto directamente DESCARTA una transicion
    BIO ilegal (no la cuenta como prediccion), mientras que `entity_spans` la cuenta
    como una entidad predicha (probablemente incorrecta) y penaliza la precision --
    se verifico esta diferencia de forma empirica antes de hacer este cambio."""
    result, active, begin, invalid = set(), None, None, 0
    for i, idx in enumerate(list(tag_ids) + [0]):
        tag = TAGS[idx] if idx >= 0 else 'O'
        prefix, label = tag.split('-', 1) if tag != 'O' else ('O', None)
        continuing = prefix == 'I' and label == active
        if active is not None and not continuing:
            result.add((begin, i, active)); active = None
        if prefix in ('B', 'I') and not continuing:
            invalid += int(prefix == 'I')
            begin, active = i, label
    return result, invalid

def scores_from_counts(tp, predicted, gold):
    """Calcula precision, recall y F1 a partir de conteos agregados: verdaderos
    positivos (tp), total de entidades predichas y total de entidades reales (gold)
    -- identica a la de taller 1."""
    precision = tp / predicted if predicted else 0.0
    recall = tp / gold if gold else 0.0
    return {'precision': precision, 'recall': recall,
            'f1': 2*precision*recall/(precision+recall) if precision+recall else 0.0,
            'support': gold, 'predicted': predicted}

class SinusoidPE(nn.Module):
    """Positional encoding sinusoidal fija (no entrenable) del paper original de
    Transformers. Se registra como buffer (no parametro) porque no se aprende."""
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)
        i = torch.arange(d_model).unsqueeze(0)
        div_term = 1 / torch.pow(10000, (2 * (i // 2)) / torch.tensor(d_model, dtype=torch.float32))
        angle_rads = pos * div_term
        pos_encoding = torch.zeros(max_len, d_model)
        pos_encoding[:, 0::2] = torch.sin(angle_rads[:, 0::2])
        pos_encoding[:, 1::2] = torch.cos(angle_rads[:, 1::2])
        self.register_buffer('pos_encoding', pos_encoding.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pos_encoding[:, :x.size(1), :]

class TokenAndPosEmbedding(nn.Module):
    """Embedding de palabra (entrenado desde cero, con `padding_idx`) mas positional
    encoding sinusoidal."""
    def __init__(self, max_len, embed_dim, vocab_size, padding_idx):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.pos_emb = SinusoidPE(max_len, embed_dim)

    def forward(self, x):
        return self.pos_emb(self.token_emb(x))

class MultiHeadAttention(nn.Module):
    """Atencion multi-cabeza escrita a mano. Corrige tres bugs de la guia de
    referencia: el assert de divisibilidad usa `%` (no `&`), las capas lineales usan
    el parametro `embed_size` (no una variable global) y `combine_heads` ya no tiene
    el typo `comibe_heads`. Tambien reemplaza el valor de mascara `-9e-15` de la guia
    (un numero casi cero, que en realidad NO suprime nada tras el softmax) por
    `-1e9`, que si deja el peso de atencion de los tokens de padding en ~0."""
    def __init__(self, embed_size, num_heads=8):
        super().__init__()
        assert embed_size % num_heads == 0, 'embed_size debe ser divisible por num_heads.'
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.projection_dim = embed_size // num_heads
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        self.combine_heads = nn.Linear(embed_size, embed_size)

    @staticmethod
    def _scaled_dot_product(q, k, v, mask=None):
        d_k = q.size()[-1]
        attn_logits = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            attn_logits = attn_logits.masked_fill(mask.reshape(mask.shape[0], 1, 1, -1) == 0, -1e9)
        attention = torch.softmax(attn_logits, dim=-1)
        return torch.matmul(attention, v), attention

    def _separate_heads(self, x, batch_size):
        x = x.reshape(batch_size, -1, self.num_heads, self.projection_dim)
        return x.permute(0, 2, 1, 3)

    def forward(self, x, mask=None, return_attention=False):
        batch_size, seq_len, embed_size = x.size()
        q, k, v = self.query(x), self.key(x), self.value(x)
        q, k, v = (self._separate_heads(t, batch_size) for t in (q, k, v))
        weights, attention = self._scaled_dot_product(q, k, v, mask)
        weights = weights.permute(0, 2, 1, 3).reshape(batch_size, seq_len, embed_size)
        output = self.combine_heads(weights)
        return (output, attention) if return_attention else output

class TransformerBlock(nn.Module):
    """Un bloque encoder de Transformer, CON conexiones residuales
    (`LayerNorm(x + Sublayer(x))`) -- la guia de referencia las omite y lo senala
    como el bug principal a corregir si se reutiliza ese codigo."""
    def __init__(self, embed_dim, num_heads=8, ff_dim=512, dropout=0.2):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.attention_dropout = nn.Dropout(dropout)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, ff_dim), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(ff_dim, embed_dim))
        self.ffn_dropout = nn.Dropout(dropout)
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        attn_output = self.attention_dropout(self.attention(x, mask))
        x = self.layer_norm1(x + attn_output)  # residual 1
        ffn_output = self.ffn_dropout(self.ffn(x))
        return self.layer_norm2(x + ffn_output)  # residual 2

class TransformerEncoder(nn.Module):
    """Apila `num_layers` bloques Transformer -- la mejora #4 que la guia de
    referencia sugiere y que aqui usamos como la variable experimental
    (`shallow`=1 bloque vs `deep`=4 bloques)."""
    def __init__(self, num_layers, embed_dim, num_heads=8, ff_dim=512, dropout=0.2):
        super().__init__()
        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, num_heads, ff_dim, dropout) for _ in range(num_layers)])

    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)
        return x

class ClinicalTransformer(nn.Module):
    """Modelo completo: embeddings + positional encoding -> N bloques Transformer ->
    un `Linear` por token. A diferencia de la guia (que clasifica todo el documento
    con `Flatten` + `Linear` gigante, el 95% de sus parametros), aqui no hace falta
    aplanar nada: cada posicion se clasifica de forma independiente, igual de simple
    que el `fc` de la Bi-LSTM en taller 1."""
    def __init__(self, vocab_size, max_len, num_labels, embed_dim, num_heads, ff_dim,
                num_layers, dropout, padding_idx):
        super().__init__()
        # El positional encoding recibe el doble de margen que max_len: en la PASO 06
        # un bloque puede crecer un poco por encima de max_length cuando una entidad
        # empieza justo en el limite del bloque y no se puede cortar a la mitad.
        # max_len (el valor real de CFG['max_length']) se preserva tal cual en
        # self.hparams para que el resto del cuaderno reporte el tamano de bloque
        # verdadero, no este margen interno.
        pe_max_len = max_len * 2
        self.token_pos_embedding = TokenAndPosEmbedding(pe_max_len, embed_dim, vocab_size, padding_idx)
        self.embedding_dropout = nn.Dropout(dropout)
        self.encoder = TransformerEncoder(num_layers, embed_dim, num_heads, ff_dim, dropout)
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding_dropout(self.token_pos_embedding(input_ids))
        x = self.encoder(x, attention_mask)
        return self.classifier(x)

class ClinicalTransformerLightning(pl.LightningModule):
    """Envoltorio de PyTorch Lightning: entrena con entropia cruzada ponderada por
    clase (ignorando IGNORE) y, al final de cada epoca de validacion, calcula F1 de
    entidad con `entity_spans`/`scores_from_counts` (coincidencia EXACTA de limites y
    categoria) -- la misma funcion de taller 1, para que el F1 final sea
    estrictamente comparable entre los tres talleres."""
    def __init__(self, vocab_size, max_len, num_labels, embed_dim, num_heads, ff_dim,
                num_layers, dropout, padding_idx, class_weights, learning_rate, weight_decay):
        super().__init__()
        self.save_hyperparameters(ignore=['class_weights'])
        self.model = ClinicalTransformer(vocab_size, max_len, num_labels, embed_dim,
                                         num_heads, ff_dim, num_layers, dropout, padding_idx)
        self.register_buffer('class_weights', class_weights)
        self.validation_step_outputs = []
        self.last_val_results = None

    def forward(self, input_ids, attention_mask=None):
        return self.model(input_ids, attention_mask)

    def _compute_loss(self, logits, labels):
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights, ignore_index=IGNORE)
        return loss_fct(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))

    def training_step(self, batch, batch_idx):
        logits = self(batch['input_ids'], batch['attention_mask'])
        loss = self._compute_loss(logits, batch['labels'])
        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        logits = self(batch['input_ids'], batch['attention_mask'])
        loss = self._compute_loss(logits, batch['labels'])
        preds = logits.argmax(-1)
        lengths = batch['attention_mask'].sum(dim=1)
        self.validation_step_outputs.append(
            (preds.detach().cpu(), batch['labels'].detach().cpu(), lengths.detach().cpu()))
        self.log('val_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def on_validation_epoch_end(self):
        # self.validation_step_outputs guarda (preds, labels, lengths) por LOTE --
        # hay que recorrer primero los lotes y despues cada fila (ejemplo) dentro del
        # lote. Igual que en taller 1: se trunca solo el padding final (via `length`)
        # y las posiciones IGNORE que quedan dentro de la secuencia se relabelan a 'O'
        # en vez de eliminarse, para no unir por accidente dos entidades separadas
        # por un hueco ignorado.
        totals = {label: Counter() for label in LABELS}
        invalid = 0
        for pred_batch, label_batch, length_batch in self.validation_step_outputs:
            for pred_row, label_row, length in zip(pred_batch.tolist(), label_batch.tolist(), length_batch.tolist()):
                g = label_row[:length]; p = pred_row[:length]
                p = [x if y != IGNORE else 0 for x, y in zip(p, g)]
                g = [y if y != IGNORE else 0 for y in g]
                ps, errors = entity_spans(p); gs, gold_errors = entity_spans(g)
                assert gold_errors == 0, 'BIO gold invalido.'
                invalid += errors
                for label in LABELS:
                    totals[label].update(tp=sum(x[2] == label for x in ps & gs),
                        predicted=sum(x[2] == label for x in ps), gold=sum(x[2] == label for x in gs))
        per_label = {k: scores_from_counts(v['tp'], v['predicted'], v['gold']) for k, v in totals.items()}
        micro = scores_from_counts(sum(v['tp'] for v in totals.values()),
                                   sum(v['predicted'] for v in totals.values()), sum(v['gold'] for v in totals.values()))
        # Mismo formato que antes ('overall_*' + un sub-diccionario por etiqueta con
        # 'f1'/'number') para no tener que tocar las PASO 10/11, que ya leen asi.
        self.last_val_results = {'overall_precision': micro['precision'], 'overall_recall': micro['recall'],
                                 'overall_f1': micro['f1'], 'invalid_bio_transitions': invalid,
                                 **{label: {'precision': per_label[label]['precision'],
                                           'recall': per_label[label]['recall'],
                                           'f1': per_label[label]['f1'],
                                           'number': per_label[label]['support']} for label in LABELS}}
        self.log('val_f1', micro['f1'], prog_bar=True)
        self.log('val_precision', micro['precision'])
        self.log('val_recall', micro['recall'])
        for label in LABELS:
            self.log(f'val_f1_{label}', per_label[label]['f1'])
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate,
                                 weight_decay=self.hparams.weight_decay)

print('PASO 08 OK. Arquitectura y LightningModule definidos; aun no se ha entrenado.')

PASO 07 OK. Arquitectura y LightningModule definidos; aun no se ha entrenado.


### PASO 09 - Prueba tecnica de las dos variantes

Antes de invertir tiempo en los 6 entrenamientos, probamos `shallow` y `deep` con un modelo temporal: comprobamos que la perdida baja al repetir un lote pequeno, que los gradientes son finitos y -- lo mas importante aqui, porque escribimos la atencion multi-cabeza a mano -- que agregar padding extra a la derecha **no cambia la salida** en las posiciones reales. Si esta PASO falla, no continues con la PASO 10.

In [61]:
# PASO 09 - PRUEBA TECNICA: gradientes finitos, la perdida baja, la mascara de padding funciona
def run_smoke_test(variant):
    """Prueba tecnica rapida (no es el entrenamiento real) antes de invertir tiempo en
    los 6 entrenamientos de la PASO 10. En un lote pequeno verifica que la perdida
    baja al repetir el mismo lote, que los gradientes son finitos y que agregar
    padding extra a la derecha NO cambia la salida en las posiciones reales (confirma
    que la mascara de atencion escrita a mano funciona). Los pesos de este modelo
    temporal se descartan; no pasan al entrenamiento real."""
    probe_batch = collate([train_dataset[i] for i in range(min(4, len(train_dataset)))])
    probe_batch = {k: v.to(device) for k, v in probe_batch.items()}
    torch.manual_seed(CFG['training_seeds'][0])
    model = ClinicalTransformerLightning(
        vocab_size=len(tokenizer), max_len=CFG['max_length'], num_labels=len(TAGS),
        embed_dim=CFG['embed_dim'], num_heads=CFG['num_heads'], ff_dim=CFG['ff_dim'],
        num_layers=CFG['variants'][variant], dropout=0.0, padding_idx=PAD_ID,
        class_weights=class_weights_tensor, learning_rate=0.01, weight_decay=0.0).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
    losses = []
    for _ in range(20):
        optimizer.zero_grad(set_to_none=True)
        logits = model(probe_batch['input_ids'], probe_batch['attention_mask'])
        loss = model._compute_loss(logits, probe_batch['labels'])
        assert torch.isfinite(loss), 'Perdida no finita en prueba tecnica.'
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        assert norm > 0, 'Gradiente nulo: revisar etiquetas.'
        optimizer.step()
        losses.append(loss.item())
    assert min(losses[-5:]) < losses[0], 'La perdida no bajo al repetir el lote.'
    model.eval()
    padding_invariant = None
    with torch.no_grad():
        original = model(probe_batch['input_ids'], probe_batch['attention_mask'])
        # El positional encoding solo tiene posiciones hasta CFG['max_length'] (el
        # tamano maximo real de un bloque); si el lote de prueba ya llega a ese tope,
        # no hay espacio para agregar padding sintetico y el chequeo se omite.
        headroom = CFG['max_length'] - probe_batch['input_ids'].shape[1]
        pad_amount = min(7, headroom)
        if pad_amount > 0:
            padded_ids = torch.nn.functional.pad(probe_batch['input_ids'], (0, pad_amount), value=PAD_ID)
            padded_mask = torch.nn.functional.pad(probe_batch['attention_mask'], (0, pad_amount), value=0)
            padded_output = model(padded_ids, padded_mask)
            torch.testing.assert_close(original, padded_output[:, :original.shape[1]], atol=1e-4, rtol=1e-3)
            padding_invariant = True
        else:
            print(f'AVISO ({variant}): el lote de prueba ya alcanza max_length; '
                  'se omite el chequeo de invariancia al padding.')
    return {'passed': True, 'steps': len(losses), 'initial_loss': losses[0], 'final_loss': losses[-1],
            'padding_invariant': padding_invariant, 'parameters': sum(p.numel() for p in model.parameters())}

try:
    smoke_results = {variant: run_smoke_test(variant) for variant in CFG['variants']}
    REPORT['smoke_test'] = {'passed': all(r['passed'] for r in smoke_results.values()), 'variants': smoke_results}
except Exception as exc:
    failure = {'status': 'ERROR', 'cell': '08', 'error': str(exc), 'traceback': traceback.format_exc()}
    (RUN / 'error.json').write_text(json.dumps(failure, indent=2), encoding='utf-8')
    print('ERROR_PARA_COMPARTIR', json.dumps(failure, indent=2), flush=True)
    raise
print('PASO 09 OK:', json.dumps(REPORT['smoke_test'], indent=2))

AVISO (shallow): el lote de prueba ya alcanza max_length; se omite el chequeo de invariancia al padding.
AVISO (deep): el lote de prueba ya alcanza max_length; se omite el chequeo de invariancia al padding.
PASO 08 OK: {
  "passed": true,
  "variants": {
    "shallow": {
      "passed": true,
      "steps": 20,
      "initial_loss": 2.5983612537384033,
      "final_loss": 0.013032212853431702,
      "padding_invariant": null,
      "parameters": 1223691
    },
    "deep": {
      "passed": true,
      "steps": 20,
      "initial_loss": 2.436117649078369,
      "final_loss": 1.9963977336883545,
      "padding_invariant": null,
      "parameters": 1818507
    }
  }
}


### PASO 10 - Seis entrenamientos: 2 variantes (shallow/deep) x 3 semillas

Entrenamos `shallow` (1 bloque) y `deep` (4 bloques) para cada semilla con `pl.Trainer`. Lightning **no** restaura automaticamente los mejores pesos en el objeto `model` al terminar `fit()` (a diferencia de Keras); por eso recargamos el mejor checkpoint (`ModelCheckpoint`) y volvemos a validar para obtener las metricas finales por categoria. Esta es la PASO que consume tiempo.

In [62]:
# PASO 10 - SEIS ENTRENAMIENTOS: 2 variantes (shallow/deep) x 3 semillas
import re

assert REPORT.get('smoke_test', {}).get('passed'), 'Ejecuta primero la PASO 09.'

def train_run(variant, training_seed):
    """Entrena una combinacion (variante, semilla) de principio a fin con PyTorch
    Lightning: reinicia la semilla, arma DataLoaders con un generador propio de esa
    semilla, entrena hasta CFG['epochs'] o hasta parada temprana por F1 de validacion
    (`EarlyStopping`), y guarda el mejor checkpoint (`ModelCheckpoint`). Como Lightning
    no restaura los mejores pesos automaticamente, recargamos el checkpoint y volvemos
    a validar para tener las metricas finales por categoria."""
    run_dir = RUN / variant / f'seed_{training_seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    pl.seed_everything(training_seed, workers=True)
    generator = torch.Generator().manual_seed(training_seed)
    train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True,
                              collate_fn=collate, generator=generator)
    val_loader = DataLoader(val_dataset, batch_size=CFG['batch_size'], shuffle=False, collate_fn=collate)
    model = ClinicalTransformerLightning(
        vocab_size=len(tokenizer), max_len=CFG['max_length'], num_labels=len(TAGS),
        embed_dim=CFG['embed_dim'], num_heads=CFG['num_heads'], ff_dim=CFG['ff_dim'],
        num_layers=CFG['variants'][variant], dropout=CFG['dropout'], padding_idx=PAD_ID,
        class_weights=class_weights_tensor, learning_rate=CFG['learning_rate'], weight_decay=CFG['weight_decay'])
    parameters = sum(p.numel() for p in model.parameters())
    checkpoint_cb = pl.callbacks.ModelCheckpoint(dirpath=str(run_dir), filename='best-{epoch}',
                                                 monitor='val_f1', mode='max', save_top_k=1)
    early_stop_cb = pl.callbacks.EarlyStopping(monitor='val_f1', mode='max', patience=CFG['patience'])
    logger = pl.loggers.CSVLogger(save_dir=str(run_dir), name='logs')
    # deterministic='warn' en vez de True: algunas operaciones de CUDA no tienen una
    # implementacion determinista y preferimos un aviso a que la corrida falle.
    trainer = pl.Trainer(max_epochs=CFG['epochs'], accelerator='gpu' if torch.cuda.is_available() else 'cpu',
                         devices=1, logger=logger, callbacks=[checkpoint_cb, early_stop_cb],
                         enable_progress_bar=True, deterministic='warn', log_every_n_steps=1)
    print(f'{variant}, semilla {training_seed}. Dispositivo: {device}, {parameters:,} parametros, '
          f'{len(train_dataset)} bloques train, {len(val_dataset)} bloques validacion, '
          f'{CFG["variants"][variant]} bloques Transformer.', flush=True)
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    # Extraemos la epoca del nombre de archivo (formato 'best-epoch=N.ckpt') en vez de
    # leer el checkpoint nosotros mismos con torch.load: dejamos que sea unicamente
    # `load_from_checkpoint` (la API de Lightning, mas abajo) quien lo cargue.
    best_epoch = int(re.search(r'epoch=(\d+)', Path(checkpoint_cb.best_model_path).stem).group(1))
    best_model = ClinicalTransformerLightning.load_from_checkpoint(
        checkpoint_cb.best_model_path, class_weights=class_weights_tensor)
    trainer.validate(best_model, dataloaders=val_loader, verbose=False)
    validation_results = best_model.last_val_results
    result = {'variant': variant, 'seed': training_seed, 'split_sha256': split_sha256,
              'parameters': parameters, 'checkpoint_path': checkpoint_cb.best_model_path,
              'best_epoch': best_epoch, 'best_metric_f1': float(checkpoint_cb.best_model_score),
              'epochs_completed': trainer.current_epoch,
              # entity_spans/scores_from_counts ya devuelven float/int nativos de
              # Python; los envolvemos en float()/int() de todas formas por defensa.
              'validation': {'micro': {'precision': float(validation_results['overall_precision']),
                                       'recall': float(validation_results['overall_recall']),
                                       'f1': float(validation_results['overall_f1']),
                                       'support': int(sum(validation_results.get(l, {}).get('number', 0) for l in LABELS))},
                             'per_label': {l: {'f1': float(validation_results.get(l, {}).get('f1', 0.0)),
                                               'support': int(validation_results.get(l, {}).get('number', 0))} for l in LABELS}}}
    (run_dir / 'result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    del trainer, model, best_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

REPORT['experiments'] = {}
n_runs = len(CFG['variants']) * len(CFG['training_seeds'])
run_number = 0
pipeline_start = time.perf_counter()
for variant in CFG['variants']:
    REPORT['experiments'][variant] = {}
    for training_seed in CFG['training_seeds']:
        run_number += 1
        print(f'\n===== Entrenamiento {run_number}/{n_runs}: variante={variant}, semilla={training_seed} =====',
              flush=True)
        run_start = time.perf_counter()
        result = train_run(variant, training_seed)
        run_minutes = (time.perf_counter() - run_start) / 60
        REPORT['experiments'][variant][str(training_seed)] = result
        f1_pct = 100 * result['validation']['micro']['f1']
        print(f'----- Entrenamiento {run_number}/{n_runs} terminado en {run_minutes:.1f} min: '
              f'mejor epoca={result["best_epoch"]}, epocas ejecutadas={result["epochs_completed"]}, '
              f'F1 validacion={f1_pct:.2f} % -----', flush=True)
        # Conservar avances si una ejecucion posterior falla.
        (RUN / 'experiments_progress.json').write_text(json.dumps(REPORT['experiments'], indent=2), encoding='utf-8')
total_minutes = (time.perf_counter() - pipeline_start) / 60
print(f'\nPASO 10 OK. Terminaron {n_runs} entrenamientos en {total_minutes:.1f} min. Ejecuta la PASO 11.')


===== Entrenamiento 1/6: variante=shallow, semilla=42 =====


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


shallow, semilla 42. Dispositivo: cuda:0, 1,223,691 parametros, 1883 bloques train, 501 bloques validacion, 1 bloques Transformer.


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/spaccc_transformer_scratch/20260919_165308_906088/shallow/seed_42 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ClinicalTransformer │  1.2 M │ train │     0 │
└───┴───────┴─────────────────────┴────────┴───────┴───────┘

...

Output()

`Trainer.fit` stopped: `max_epochs=30` reached.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

----- Entrenamiento 6/6 terminado en 2.2 min: mejor epoca=29, epocas ejecutadas=30, F1 validacion=25.54 % -----

PASO 09 OK. Terminaron 6 entrenamientos en 9.7 min. Ejecuta la PASO 10.


### PASO 11 - Agregacion y tabla comparativa final

Resumimos los 6 entrenamientos sin volver a entrenar. Calculamos F1/precision/recall por semilla, medias y desviaciones estandar por variante, la diferencia pareada `deep - shallow`, y F1 por categoria. La tabla `aggregate` es la comparacion final que va en el informe -- responde la pregunta de la mejora #4 de la guia: ?ayuda apilar mas bloques Transformer cuando se entrena desde cero con pocos datos?

In [63]:
# PASO 11 - MEDIA, DESVIACION ESTANDAR, DIFERENCIA PAREADA Y RESUMEN
experiments = REPORT.get('experiments', {})
assert set(experiments) == set(CFG['variants']), 'Faltan variantes: completa la PASO 10.'
assert all(set(pair) == {str(s) for s in CFG['training_seeds']} for pair in experiments.values()), 'Faltan semillas: completa la PASO 10.'

rows, label_rows = [], []
gold_support = None
for variant in CFG['variants']:
    for seed in CFG['training_seeds']:
        result = experiments[variant][str(seed)]
        assert result['seed'] == seed and result['split_sha256'] == REPORT['split']['sha256']
        metrics = result['validation']['micro']
        if gold_support is None: gold_support = metrics['support']
        assert metrics['support'] == gold_support, 'Las ejecuciones no evaluan el mismo gold.'
        rows.append({'semilla': seed, 'variante': variant, 'parametros': result['parameters'],
            'mejor_epoca': result['best_epoch'], 'epocas_ejecutadas': result['epochs_completed'],
            'precision_pct': 100*metrics['precision'], 'recall_pct': 100*metrics['recall'],
            'f1_pct': 100*metrics['f1']})
        for label in LABELS:
            m = result['validation']['per_label'][label]
            label_rows.append({'semilla': seed, 'variante': variant, 'categoria': label,
                               'entidades_validacion': m['support'], 'f1_pct': 100*m['f1']})
comparison = pd.DataFrame(rows)
per_label = pd.DataFrame(label_rows)
assert (per_label.groupby('categoria')['entidades_validacion'].nunique() == 1).all()

# Desviacion estandar muestral (ddof=1): variabilidad de las semillas, no intervalo de confianza.
aggregate_rows = []
for name, group in comparison.groupby('variante', sort=False):
    row = {'variante': name, 'n_semillas': len(group)}
    for metric in ['precision_pct', 'recall_pct', 'f1_pct']:
        row[metric + '_media'] = float(group[metric].mean())
        row[metric + '_std'] = float(group[metric].std(ddof=1))
    aggregate_rows.append(row)
aggregate = pd.DataFrame(aggregate_rows)

# Diferencia pareada: deep - shallow, misma semilla.
by_seed = comparison.pivot(index='semilla', columns='variante', values='f1_pct')
delta = by_seed['deep'] - by_seed['shallow']
paired_rows = [{'semilla': seed, 'f1_deep_pct': by_seed.loc[seed, 'deep'],
               'f1_shallow_pct': by_seed.loc[seed, 'shallow'], 'diferencia_pp': value}
              for seed, value in delta.items()]
paired = pd.DataFrame(paired_rows)

label_aggregate_rows = []
for variant in CFG['variants']:
    for label in LABELS:
        group = per_label[(per_label.variante == variant) & (per_label.categoria == label)]
        label_aggregate_rows.append({'variante': variant, 'categoria': label,
            'entidades_validacion': int(group.entidades_validacion.iloc[0]),
            'f1_media_pct': float(group.f1_pct.mean()), 'f1_std_pct': float(group.f1_pct.std(ddof=1))})
label_aggregate = pd.DataFrame(label_aggregate_rows)

for name, table in [('comparison_runs', comparison), ('comparison_aggregate', aggregate),
                    ('comparison_paired', paired), ('comparison_per_label_aggregate', label_aggregate)]:
    table.to_csv(RUN / f'{name}.csv', index=False)

delta_mean = float(delta.mean())
REPORT['comparison'] = {'per_run': rows, 'aggregate': aggregate_rows, 'paired': paired_rows,
    'per_label_aggregate': label_aggregate_rows,
    'paired_summary': {'delta_f1_mean_pp_deep_minus_shallow': delta_mean,
                       'deep_wins': int((delta > 1e-10).sum()), 'shallow_wins': int((delta < -1e-10).sum())},
    'note': 'Media y desviacion muestral entre semillas de entrenamiento, con una particion fija. No es validacion cruzada ni una prueba de significancia estadistica.'}
REPORT['elapsed_seconds_since_cell02'] = time.perf_counter() - START
REPORT['status'] = 'COMPARACION_SEIS_ENTRENAMIENTOS_COMPLETADA'
REPORT['metric_scope'] = 'Entidades exactas BIO (entity_spans/scores_from_counts, misma funcion que taller 1).'
(RUN / 'summary.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')

print('TABLA COMPARATIVA FINAL (F1 medio +/- desviacion estandar, en %):')
print(aggregate.round(3).to_string(index=False))
print()
print('DIFERENCIA deep - shallow (por semilla):')
print(paired.round(3).to_string(index=False))
print()
print('F1 POR CATEGORIA:')
print(label_aggregate.round(3).to_string(index=False))
print()
print('Archivos guardados en:', RUN)
print('Pega la tabla comparativa (aggregate) en el informe final de este cuaderno.')

TABLA COMPARATIVA FINAL (F1 medio +/- desviacion estandar, en %):
variante  n_semillas  precision_pct_media  precision_pct_std  recall_pct_media  recall_pct_std  f1_pct_media  f1_pct_std
 shallow           3               20.799              0.345            29.829           1.200        24.506       0.638
    deep           3               21.373              0.766            30.716           1.826        25.201       1.100

DIFERENCIA deep - shallow (por semilla):
 semilla  f1_deep_pct  f1_shallow_pct  diferencia_pp
      42       26.095          24.695          1.400
     123       23.973          23.795          0.179
    2026       25.536          25.028          0.508

F1 POR CATEGORIA:
variante categoria  entidades_validacion  f1_media_pct  f1_std_pct
 shallow  CHEMICAL                   589        28.007       1.425
 shallow   DISEASE                  1395        20.018       0.309
 shallow PROCEDURE                  1909        29.213       1.158
 shallow   PROTEIN            

### PASO 12 - Evaluacion final en test (opcional, desactivada)

Igual que en talleres 1 y 3: el conjunto de test se mantiene reservado por defecto. Solo se evalua si activas `run_final_test=True` y eliges explicitamente `final_test_variant` y `final_test_seed` despues de revisar la tabla comparativa de la PASO 11.

In [64]:
# PASO 12 - TEST FINAL OPCIONAL DE UNA VARIANTE ELEGIDA; DESACTIVADO
if not CFG['run_final_test']:
    print('PASO 12 OMITIDA: test reservado. Comparacion terminada en la PASO 11.')
elif CFG['fraction'] != 1.0:
    raise RuntimeError('Test bloqueado: completa primero el entrenamiento con fraction=1.0.')
elif CFG['final_test_variant'] not in CFG['variants']:
    raise RuntimeError(f'Selecciona explicitamente final_test_variant entre {list(CFG["variants"])}.')
elif CFG['final_test_seed'] not in CFG['training_seeds']:
    raise RuntimeError('Selecciona explicitamente final_test_seed entre las semillas entrenadas.')
else:
    variant = CFG['final_test_variant']
    seed = CFG['final_test_seed']
    result = REPORT['experiments'][variant][str(seed)]
    best_model = ClinicalTransformerLightning.load_from_checkpoint(
        result['checkpoint_path'], class_weights=class_weights_tensor)
    test_examples, test_stats = make_examples(official_test_ids, df_test, 'test')
    test_loader = DataLoader(ClinicalDataset(test_examples), batch_size=CFG['batch_size'],
                             shuffle=False, collate_fn=collate)
    test_trainer = pl.Trainer(accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
                              logger=False, enable_progress_bar=False)
    test_trainer.validate(best_model, dataloaders=test_loader, verbose=False)
    test_results = best_model.last_val_results
    # entity_spans/scores_from_counts ya devuelven float/int nativos de Python.
    final_test = {'variant': variant, 'seed': seed,
                 'metrics': {'micro': {'precision': float(test_results['overall_precision']),
                                       'recall': float(test_results['overall_recall']),
                                       'f1': float(test_results['overall_f1'])},
                            'per_label': {l: {'f1': float(test_results.get(l, {}).get('f1', 0.0)),
                                              'support': int(test_results.get(l, {}).get('number', 0))} for l in LABELS}},
                 'alignment': test_stats, 'checkpoint_path': result['checkpoint_path'],
                 'metric_scope': REPORT['metric_scope']}
    (RUN / f'final_test_{variant}_seed_{seed}.json').write_text(json.dumps(final_test, indent=2), encoding='utf-8')
    print('RESULTADO_TEST_FINAL', json.dumps(final_test, indent=2))

PASO 11 OMITIDA: test reservado. Comparacion terminada en la PASO 10.


### PASO 13 - Identificacion interactiva en un texto nuevo

Como no usamos `pipeline` de Hugging Face (nuestro modelo es 100% propio), reconstruimos las entidades a mano igual que taller 1: tokenizamos con nuestro tokenizer BPE, predecimos etiquetas BIO token por token y las agrupamos con `entity_spans`.

In [65]:
# PASO 13 - IDENTIFICACION INTERACTIVA EN UN TEXTO NUEVO
TEXT_TO_IDENTIFY = ("Paciente con cefalea intensa y fiebre. Se indico tratamiento "
                    "con paracetamol y se solicito una resonancia magnetica.")
IDENTIFICATION_VARIANT = 'deep'    # 'shallow' o 'deep'
IDENTIFICATION_SEED = 42           # una semilla de CFG['training_seeds']

if not isinstance(TEXT_TO_IDENTIFY, str) or not TEXT_TO_IDENTIFY.strip():
    raise ValueError('TEXT_TO_IDENTIFY debe ser una cadena no vacia.')
if IDENTIFICATION_VARIANT not in CFG['variants']:
    raise ValueError(f'IDENTIFICATION_VARIANT debe pertenecer a {list(CFG["variants"])}.')
if IDENTIFICATION_SEED not in CFG['training_seeds']:
    raise ValueError(f'IDENTIFICATION_SEED debe pertenecer a {CFG["training_seeds"]}.')
experiments = REPORT.get('experiments', {})
combo = experiments.get(IDENTIFICATION_VARIANT, {})
if str(IDENTIFICATION_SEED) not in combo:
    raise RuntimeError('No existe el resultado de esa combinacion; ejecuta primero la PASO 10.')

result = combo[str(IDENTIFICATION_SEED)]
inference_model = ClinicalTransformerLightning.load_from_checkpoint(
    result['checkpoint_path'], class_weights=class_weights_tensor)
inference_model.to(device).eval()

def entity_spans(tag_ids):
    """Agrupa una secuencia de indices de etiqueta BIO en entidades (inicio, fin,
    categoria) -- misma logica que taller 1 y taller 3: una I- sin entidad activa se
    trata como inicio de una nueva entidad (reparacion deterministica), y se cuenta
    cuantas veces ocurrio."""
    result_set, active, begin, invalid = set(), None, None, 0
    for i, idx in enumerate(list(tag_ids) + [0]):
        tag = TAGS[idx] if idx >= 0 else 'O'
        prefix, label = tag.split('-', 1) if tag != 'O' else ('O', None)
        continuing = prefix == 'I' and label == active
        if active is not None and not continuing:
            result_set.add((begin, i, active)); active = None
        if prefix in ('B', 'I') and not continuing:
            invalid += int(prefix == 'I')
            begin, active = i, label
    return result_set, invalid

encoding = tokenizer(TEXT_TO_IDENTIFY, add_special_tokens=False, return_offsets_mapping=True)
input_ids, offset_mapping = encoding['input_ids'], encoding['offset_mapping']
if len(input_ids) == 0:
    raise ValueError('El tokenizer no encontro tokens en TEXT_TO_IDENTIFY.')

all_predictions = []
with torch.no_grad():
    for start in range(0, len(input_ids), CFG['max_length']):
        end = min(start + CFG['max_length'], len(input_ids))
        chunk_ids = torch.tensor([input_ids[start:end]], dtype=torch.long, device=device)
        chunk_mask = torch.ones_like(chunk_ids)
        logits = inference_model(chunk_ids, chunk_mask)[0]
        probabilities = torch.softmax(logits, dim=-1)
        scores, indices = probabilities.max(dim=-1)
        all_predictions.extend(zip(offset_mapping[start:end], indices.cpu().tolist(), scores.cpu().tolist()))

token_rows = [{'start': s, 'end': e, 'tag': TAGS[idx], 'confidence': round(score, 4), 'is_entity': TAGS[idx] != 'O'}
             for (s, e), idx, score in all_predictions]
predicted_ids = [idx for _, idx, _ in all_predictions]
predicted_entities, invalid_transitions = entity_spans(predicted_ids)
entities = []
for start, end, label in sorted(predicted_entities):
    selected = all_predictions[start:end]
    if not selected:
        continue
    first_offset = selected[0][0]
    last_offset = selected[-1][0]
    entities.append({'text': TEXT_TO_IDENTIFY[first_offset[0]:last_offset[1]], 'label': label,
                     'start': first_offset[0], 'end': last_offset[1],
                     'confidence_mean': round(float(np.mean([score for _, _, score in selected])), 4)})

identification = {'text': TEXT_TO_IDENTIFY, 'variant': IDENTIFICATION_VARIANT, 'training_seed': IDENTIFICATION_SEED,
                  'checkpoint_path': result['checkpoint_path'], 'tokens': token_rows, 'entities': entities,
                  'invalid_bio_transitions': invalid_transitions,
                  'warning': 'La confianza no esta calibrada; las predicciones no sustituyen evaluacion clinica.'}
print('========== IDENTIFICACION ==========', flush=True)
print(json.dumps(identification, ensure_ascii=False, indent=2), flush=True)
print('=====================================', flush=True)
print(f"Entidades detectadas: {len(entities)} | Tokens: {len(token_rows)} | "
      f"Transiciones BIO invalidas reparadas: {invalid_transitions}", flush=True)

========== IDENTIFICACION ==========
{
  "text": "Paciente con cefalea intensa y fiebre. Se indico tratamiento con paracetamol y se solicito una resonancia magnetica.",
  "variant": "deep",
  "training_seed": 42,
  "checkpoint_path": "/kaggle/working/spaccc_transformer_scratch/20260919_165308_906088/deep/seed_42/best-epoch=29.ckpt",
  "tokens": [
    {
      "start": 0,
      "end": 8,
      "tag": "O",
      "confidence": 0.968,
      "is_entity": false
    },
    {
      "start": 8,
      "end": 12,
      "tag": "O",
      "confidence": 0.7136,
      "is_entity": false
    },
    {
      "start": 12,
      "end": 20,
      "tag": "B-SYMPTOM",
      "confidence": 0.99,
      "is_entity": true
    },
    {
      "start": 20,
      "end": 28,
      "tag": "I-SYMPTOM",
      "confidence": 0.3578,
      "is_entity": true
    },
    {
      "start": 28,
      "end": 30,
      "tag": "O",
      "confidence": 0.6434,
      "is_entity": false
    },
    {
      "start": 30,
      "end": 37,
 

# Informe final - NER clinico en espanol con un Transformer desde cero

## Objetivo y diseno

Comparamos dos profundidades de un Transformer encoder implementado desde cero (sin pesos preentrenados, tokenizer BPE propio entrenado sobre SPACCC): **`shallow`** (1 bloque Transformer) y **`deep`** (4 bloques), siguiendo la arquitectura de la guia de referencia `1-transformers-from-scratch.ipynb` adaptada de clasificacion de documentos a NER por token. Usamos la misma particion de documentos de talleres 1 y 3 (`split_seed=42`, huella `ab06b4e8...`, 600 documentos de entrenamiento, 150 de validacion, 250 de test reservados) y las mismas tres semillas de entrenamiento (42, 123, 2026).

Configuracion del modelo: `embed_dim=128`, `num_heads=8`, `ff_dim=512`, `dropout=0.2`, vocabulario BPE propio de 8,000 tokens (entrenado solo sobre los 600 documentos de train), bloques de hasta 192 subwords, `batch_size=16`, `learning_rate=3e-4`, hasta 30 epocas con parada temprana (`patience=5`). `shallow` tiene 1,223,691 parametros; `deep`, 1,818,507. Las 6 corridas (2 variantes x 3 semillas) tardaron 9.7 minutos en total en 1 GPU.

## Nota metodologica: bug de alineacion corregido antes de esta ejecucion

Una primera corrida detecto que nuestro tokenizer BPE byte-level (misma familia que GPT-2/RoBERTa) reporta el offset de cada palabra incluyendo el espacio previo, mientras que las anotaciones de SPACCC empiezan en la primera letra -- eso hacia fallar la alineacion estricta en el 96% de los casos. Se corrigio `resolve_gold` (recorta los espacios iniciales del rango de cada subword antes de compararlo) y se verifico contra el corpus real. **Esta ejecucion ya incluye la correccion**; los numeros de abajo son los definitivos de este taller.

## Nota sobre la metrica de evaluacion (cambio posterior a esta ejecucion)

**Los numeros de este informe se calcularon con `seqeval` (modo estricto).** Despues de esta corrida encontramos que `seqeval` en modo estricto **descarta** las transiciones BIO ilegales de una prediccion en vez de contarlas como una entidad predicha (incorrecta); la funcion `entity_spans` de taller 1 si las cuenta, penalizando la precision. Verificamos esta diferencia de forma empirica y reemplazamos `seqeval` por `entity_spans`/`scores_from_counts` (PASO 08) para que el F1 sea estrictamente comparable con taller 1 y taller 3.

Esto significa que **las cifras de F1/precision de este informe son ligeramente optimistas** frente a lo que arrojaria el mismo modelo evaluado con `entity_spans` (el ejemplo de la PASO 13 ya muestra 4 transiciones BIO invalidas en un solo texto de prueba, asi que el efecto no es despreciable). Los analisis y comparaciones de abajo siguen siendo utiles para entender la arquitectura y la metodologia, pero **hay que volver a ejecutar las PASO 09-11 con la metrica corregida antes de dar estos numeros por definitivos**.

## Calidad de la alineacion (PASO 06)

| Particion | Anotaciones | Sin alinear | Solapes excluidos | Retenidas |
|---|---:|---:|---:|---:|
| Entrenamiento | 26,849 | 77 (0.29 %) | 4,286 | 22,486 (83.7 %) |
| Validacion | 6,908 | 25 (0.36 %) | 986 | 5,897 (85.4 %) |

Practicamente identico a taller 1 (22,401/26,849 train y 5,861/6,908 validacion retenidas) -- confirma que la comparacion entre talleres es valida sobre un conjunto de entidades equivalente.

## Resultado de validacion multisemilla

| Variante | F1 medio | Desv. estandar | Precision media | Recall medio |
|---|---:|---:|---:|---:|
| shallow | 24.51 % | 0.64 % | 20.80 % | 29.83 % |
| deep | 25.20 % | 1.10 % | 21.37 % | 30.72 % |

## Diferencia pareada deep - shallow

| Semilla | F1 deep | F1 shallow | Diferencia |
|---:|---:|---:|---:|
| 42 | 26.10 % | 24.70 % | +1.40 pp |
| 123 | 23.97 % | 23.80 % | +0.18 pp |
| 2026 | 25.54 % | 25.03 % | +0.51 pp |

`deep` supero a `shallow` en las 3 semillas, con una mejora media de **+0.70 puntos porcentuales**. La ventaja es consistente pero modesta -- de magnitud similar a la variabilidad entre semillas de cada variante (desviaciones estandar de 0.64-1.10 pp).

## F1 medio por categoria

| Categoria | Entidades en validacion | F1 shallow | F1 deep |
|---|---:|---:|---:|
| CHEMICAL | 589 | 28.01 % | 30.13 % |
| DISEASE | 1,395 | 20.02 % | 21.90 % |
| PROCEDURE | 1,909 | 29.21 % | 28.01 % |
| PROTEIN | 327 | 33.83 % | 37.54 % |
| SYMPTOM | 1,677 | 19.15 % | 20.42 % |

`deep` gana en 4 de 5 categorias; `PROCEDURE` es la unica donde `shallow` rinde ligeramente mejor.

## Comparacion con taller 1 (Bi-LSTM, misma particion)

| Modelo | F1 medio en validacion |
|---|---:|
| Bi-LSTM con POS (taller 1) | 38.96 % |
| Bi-LSTM sin POS (taller 1) | 37.13 % |
| Transformer desde cero, shallow (taller 2) | 24.51 % |
| Transformer desde cero, deep (taller 2) | 25.20 % |

El Transformer desde cero queda **13-14 puntos porcentuales por debajo** de la Bi-LSTM de taller 1, a pesar de usar practicamente el mismo conjunto de entidades. Es la confirmacion practica de la propia conclusion de la guia de referencia: los transformers entrenados desde cero son "hambrientos de datos", y 600 documentos con un vocabulario BPE de solo 8,000 tokens no alcanzan para que la arquitectura aprenda representaciones tan utiles como una Bi-LSTM mas simple (que ademas se apoya en POS tagging simbolico).

## Ejemplo de inferencia (PASO 13)

Sobre "Paciente con cefalea intensa y fiebre. Se indico tratamiento con paracetamol y se solicito una resonancia magnetica.", el modelo `deep` (semilla 42) detecto 10 entidades. Identifico correctamente **cefalea intensa** y **fiebre** como SYMPTOM, **paracetamol** como CHEMICAL y **resonancia** como PROCEDURE (confianzas altas, 0.74-1.00). Tambien produjo fragmentos claramente erroneos: partio "indico" en "in" (DISEASE, confianza 0.21) y "dico" (CHEMICAL), "tratamiento" fue correcto pero "magnetica" se partio en "magne" (CHEMICAL) y "tica" (SYMPTOM); ademas reparo 4 transiciones BIO invalidas. Estos fragmentos ilegibles reflejan el vocabulario BPE pequeno (8,000 tokens sobre un corpus chico): palabras poco frecuentes se parten en pedazos que el modelo etiqueta con poca coherencia.

## Evaluacion final en test

No se ejecuto (`run_final_test=False`); sigue reservado, igual que en talleres 1 y 3.

## Limitaciones y alcance

- El vocabulario BPE propio (8,000 tokens, entrenado solo sobre 600 documentos) es mucho mas pequeno y menos robusto que el vocabulario subword de un modelo preentrenado (taller 3) o el vocabulario de palabras completas de taller 1; probablemente explica buena parte de la brecha de F1 frente a ambos.
- Esquema BIO plano (anotaciones anidadas o solapadas se excluyen: 4,286 en train, 986 en validacion).
- Particion fija con tres semillas: la desviacion estandar describe variabilidad de entrenamiento, no significancia estadistica.
- La confianza de la PASO 13 no esta calibrada y las predicciones no sustituyen una decision clinica.
- Una ejecucion previa de este mismo cuaderno tenia un bug de alineacion (tokenizer byte-level incluyendo el espacio previo en el offset de cada palabra) que descartaba el 96% de las entidades; ya esta corregido y esta ejecucion lo incluye.

## Conclusion

**Provisional, pendiente de la re-ejecucion con `entity_spans` (ver nota de metrica arriba).** Con la alineacion corregida y un conjunto de entidades equivalente al de taller 1, `deep` (4 bloques) supero de forma consistente pero modesta a `shallow` (1 bloque) en las 3 semillas (+0.70 pp de F1 en promedio) -- apilar mas bloques Transformer ayudo un poco incluso con un corpus de este tamano, aunque el margen es chico frente a la variabilidad entre semillas. Frente a los otros talleres, el Transformer desde cero rindio claramente por debajo de la Bi-LSTM de taller 1 (24.5-25.2 % vs 37-39 % de F1), consistente con la conclusion de la guia de referencia: sin pesos preentrenados, un Transformer necesita mucho mas texto del que ofrece SPACCC para superar a arquitecturas mas simples. Es razonable esperar que ambas conclusiones (ventaja de `deep` y brecha frente a taller 1) se mantengan cualitativamente con la metrica corregida, aunque los valores exactos de F1 deben confirmarse.

Artefactos: `/kaggle/working/spaccc_transformer_scratch/20260919_165308_906088` (checkpoints por variante y semilla, `summary.json`, tablas `comparison_*.csv`, logs de entrenamiento por corrida).